# 02. 머신러닝 예측 모델 (Machine Learning Model)
월세는 통학거리 뿐만 아니라 방의 '임대면적'과 '건축년도'에도 큰 영향을 받습니다. 넓고 신축인 방과 좁고 오래된 방의 가격이 섞여있으면 거리만의 순수한 상관관계를 분석할 수 없습니다.

따라서 임대면적 등 다른 요소를 통제한 상태에서 **'거리'와 '월세' 간의 순수한 부분 의존도(Partial Dependence)**를 파악하기 위해 `HistGradientBoosting` 회귀 모델을 학습시킵니다.

**[제약조건]**
- 임대면적: 클수록 무조건 월세가 비싸져야 함 (Monotonic Constraint: +1)
- 건축년도: 최신(숫자가 클수록)일수록 무조건 월세가 비싸져야 함 (Monotonic Constraint: +1)
- 통학거리: 모델이 자율적으로 비선형적 상권 특성을 파악하도록 제약 해제 (Monotonic Constraint: 0)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import joblib
import os

# 전처리된 최종 데이터 로드
df = pd.read_csv("data/processed/final_rent_data.csv")
df = df.dropna(subset=['임대면적', '직선거리(m)', '건축년도', '월실질주거비(만원)'])

# 학습에 사용할 피처 (면적, 거리, 년도)
features = ["임대면적", "직선거리(m)", "건축년도"]
target = "월실질주거비(만원)"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 단조 제약 조건 부여: 임대면적(+1), 직선거리(0), 건축년도(+1)
model = HistGradientBoostingRegressor(
    max_iter=200, 
    max_depth=10, 
    random_state=42,
    monotonic_cst=[1, 0, 1] 
)

print("모델 학습 진행 중...")
model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"✅ 학습 완료! (평균 오차: {rmse:.2f} 만원)\n")

# 03번 분석 노트북에서 사용하기 위해 학습된 모델을 저장합니다.
os.makedirs("data/processed", exist_ok=True)
joblib.dump(model, "data/processed/rent_model.pkl")
print("모델이 'data/processed/rent_model.pkl'로 저장되었습니다.")
